# Connect IDE Clients to MCP Servers

Configure your IDE to connect to the MCP servers deployed on OpenShift via **Direct Route** URLs (no auth required).

> **Workbench users:** This notebook describes configuring your **local IDE** (VS Code, Cursor, Claude Code) on your developer workstation. If you are running this lab in an RHOAI Workbench, use this notebook as a reference guide and apply the settings on your local machine.

> For the **MaaS Gateway unified endpoint** (single URL, API key auth), see `../3_basic_run/1_ide_model_config.ipynb` §4 (MaaS Gateway unified endpoint).

**Supported IDEs:** Cursor, VS Code (Agent Mode), Claude Code, OpenCode

### Prerequisites: Self-Signed Certificate

OpenShift Routes use cluster CA certificates that IDEs (Node.js-based) do not trust by default.
Run this **once** in any terminal:

```bash
# macOS — sets env var for all GUI apps (Dock, Spotlight, terminal)
launchctl setenv NODE_TLS_REJECT_UNAUTHORIZED 0

# Linux — add to ~/.bashrc
echo 'export NODE_TLS_REJECT_UNAUTHORIZED=0' >> ~/.bashrc && source ~/.bashrc
```

Then **quit (`Cmd+Q`) and reopen** your IDE — no need to launch from terminal.

> To revert: `launchctl setenv NODE_TLS_REJECT_UNAUTHORIZED 1` + restart IDE.

## 1. Discover MCP Server Routes

In [3]:
import subprocess, json

result = subprocess.run(
    ["oc", "get", "ingresses.config", "cluster", "-o", "jsonpath={.spec.domain}"],
    capture_output=True, text=True
)
CLUSTER_DOMAIN = result.stdout.strip()

routes_result = subprocess.run(
    ["oc", "get", "routes", "-n", "mcp-servers",
     "-o", "jsonpath={range .items[*]}{.metadata.name}={.spec.host}\n{end}"],
    capture_output=True, text=True
)

mcp_urls = {}
for line in routes_result.stdout.strip().split("\n"):
    if "=" in line:
        name, host = line.split("=", 1)
        short_name = name.replace("mcp-", "")
        mcp_urls[short_name] = f"https://{host}/mcp"

print(f"Cluster domain: {CLUSTER_DOMAIN}")
print(f"Discovered {len(mcp_urls)} MCP servers:")
print("")
for name, url in sorted(mcp_urls.items()):
    print(f"  {name:<20} {url}")

Cluster domain: apps.openshift-cluster.sandbox1785.opentlc.com
Discovered 6 MCP servers:

  code-sandbox         https://mcp-code-sandbox-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp
  codebase-search      https://mcp-codebase-search-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp
  context7             https://mcp-context7-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp
  searxng           https://mcp-searxng-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp
  ocp-server           https://ocp-mcp-server-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp
  repo-docs            https://mcp-repo-docs-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp


Expected output (5 servers):
```
code-sandbox         https://mcp-code-sandbox-mcp-servers.apps.<domain>/mcp
codebase-search      https://mcp-codebase-search-mcp-servers.apps.<domain>/mcp
context7             https://mcp-context7-mcp-servers.apps.<domain>/mcp
searxng           https://mcp-searxng-mcp-servers.apps.<domain>/mcp
repo-docs            https://mcp-repo-docs-mcp-servers.apps.<domain>/mcp
```

## 2. Cursor IDE

Create `.cursor/mcp.json` in your project root:

In [4]:
cursor_config = {"mcpServers": {}}

for name, url in sorted(mcp_urls.items()):
    cursor_config["mcpServers"][name] = {"url": url}

print("=== .cursor/mcp.json ===")
print(json.dumps(cursor_config, indent=2))
print("\nCommit this file to your repo — all team members get the same MCP tools automatically.")

=== .cursor/mcp.json ===
{
  "mcpServers": {
    "code-sandbox": {
      "url": "https://mcp-code-sandbox-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp"
    },
    "codebase-search": {
      "url": "https://mcp-codebase-search-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp"
    },
    "context7": {
      "url": "https://mcp-context7-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp"
    },
    "searxng": {
      "url": "https://mcp-searxng-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp"
    },
    "ocp-server": {
      "url": "https://ocp-mcp-server-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp"
    },
    "repo-docs": {
      "url": "https://mcp-repo-docs-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp"
    }
  }
}

Commit this file to your repo — all team members get the same MCP tools automatically.


### Registered MCP servers for Cursor

![Cursor MCP Servers](../images/cursor_registered.png)

## 3. VS Code (Agent Mode)

Create `.vscode/mcp.json` in your project root (requires VS Code 1.100+):

In [5]:
vscode_config = {"servers": {}}

for name, url in sorted(mcp_urls.items()):
    vscode_config["servers"][name] = {
        "type": "http",
        "url": url
    }

print("=== .vscode/mcp.json ===")
print(json.dumps(vscode_config, indent=2))

=== .vscode/mcp.json ===
{
  "servers": {
    "code-sandbox": {
      "type": "http",
      "url": "https://mcp-code-sandbox-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp"
    },
    "codebase-search": {
      "type": "http",
      "url": "https://mcp-codebase-search-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp"
    },
    "context7": {
      "type": "http",
      "url": "https://mcp-context7-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp"
    },
    "searxng": {
      "type": "http",
      "url": "https://mcp-searxng-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp"
    },
    "ocp-server": {
      "type": "http",
      "url": "https://ocp-mcp-server-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp"
    },
    "repo-docs": {
      "type": "http",
      "url": "https://mcp-repo-docs-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp"
    }
  }
}


### Registered MCP servers for VS Code

![VS Code MCP Servers](../images/vscode_registered.png)

## 4. Claude Code

Create `.mcp.json` in project root (or `~/.claude/mcp.json` for global):

In [6]:
claude_config = {"mcpServers": {}}

for name, url in sorted(mcp_urls.items()):
    claude_config["mcpServers"][name] = {"type": "http", "url": url}

print("=== .mcp.json (Claude Code) ===")
print(json.dumps(claude_config, indent=2))
print("\nAlternatively, add via CLI:")
for name, url in sorted(mcp_urls.items()):
    print(f"  claude mcp add --transport http {name} \"{url}\"")

=== .mcp.json (Claude Code) ===
{
  "mcpServers": {
    "code-sandbox": {
      "type": "http",
      "url": "https://mcp-code-sandbox-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp"
    },
    "codebase-search": {
      "type": "http",
      "url": "https://mcp-codebase-search-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp"
    },
    "context7": {
      "type": "http",
      "url": "https://mcp-context7-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp"
    },
    "searxng": {
      "type": "http",
      "url": "https://mcp-searxng-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp"
    },
    "ocp-server": {
      "type": "http",
      "url": "https://ocp-mcp-server-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp"
    },
    "repo-docs": {
      "type": "http",
      "url": "https://mcp-repo-docs-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp"
    }
  }
}

Alternatively, add via CLI:
  claude mcp

### Registered MCP servers for Claude Code

![Claude Code MCP Servers](../images/claudecode_regiestered.png)

## 5. OpenCode

Create `opencode.json` in your project root (or `~/.config/opencode/opencode.json` for global):

> **Install OpenCode:** `curl -fsSL https://opencode.ai/install | bash`
>
> OpenCode uses `type: "remote"` for HTTP-based MCP servers. The config key is `"mcp"` (not `"mcpServers"`).

> **Self-signed certificate (OpenCode-specific):** OpenCode is a terminal CLI, so `launchctl setenv` alone may not apply. Ensure the env var is set in your **shell session**:
> ```bash
> # Add to ~/.zshrc (macOS) or ~/.bashrc (Linux)
> export NODE_TLS_REJECT_UNAUTHORIZED=0
> ```
> Or launch with: `NODE_TLS_REJECT_UNAUTHORIZED=0 opencode`
>
> **MCP timeout:** OpenCode's default MCP timeout is 5 seconds. External servers (Context7) and AI-powered servers (Codebase Search, Repo Docs) need longer — set `"timeout": 30000` (30s) to prevent them from being disabled on startup.

In [5]:
SLOW_SERVERS = {"context7", "codebase-search", "repo-docs"}

opencode_config = {
    "$schema": "https://opencode.ai/config.json",
    "mcp": {}
}

for name, url in sorted(mcp_urls.items()):
    entry = {"type": "remote", "url": url}
    if name in SLOW_SERVERS:
        entry["timeout"] = 30000
    opencode_config["mcp"][name] = entry

print("=== opencode.json ===")
print(json.dumps(opencode_config, indent=2))
print("\nPlace in your project root. OpenCode reads this on startup.")
print("Default MCP timeout is 5s; Context7 and AI servers need longer (30s).")

=== opencode.json ===
{
  "$schema": "https://opencode.ai/config.json",
  "mcp": {
    "code-sandbox": {
      "type": "remote",
      "url": "https://mcp-code-sandbox-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp"
    },
    "codebase-search": {
      "type": "remote",
      "url": "https://mcp-codebase-search-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp",
      "timeout": 30000
    },
    "context7": {
      "type": "remote",
      "url": "https://mcp-context7-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp",
      "timeout": 30000
    },
    "searxng": {
      "type": "remote",
      "url": "https://mcp-searxng-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp"
    },
    "ocp-server": {
      "type": "remote",
      "url": "https://ocp-mcp-server-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp"
    },
    "repo-docs": {
      "type": "remote",
      "url": "https://mcp-repo-docs-mcp-servers.apps.openshift

### Registered MCP servers for OpenCode

![OpenCode MCP Servers](../images/opencode_added_mcp.png)

## 6. Team Deployment Tips

### Shared Project Config

Commit the MCP config to your project repository so all team members get the same tools:

```bash
# For VS Code teams
git add .vscode/mcp.json
git commit -m "Add shared MCP server configuration"

# For Cursor teams
git add .cursor/mcp.json
git commit -m "Add shared MCP server configuration"

# For OpenCode teams
git add opencode.json
git commit -m "Add shared MCP server configuration"
```

### DNS Alias (Optional)

For cleaner URLs, create a Route with a custom hostname:

```yaml
apiVersion: route.openshift.io/v1
kind: Route
metadata:
  name: mcp-tools-custom
  namespace: mcp-servers
spec:
  host: mcp-tools.company.com
  to:
    kind: Service
    name: mcp-code-sandbox
  tls:
    termination: edge
```

## 7. Verification

In [5]:
print("MCP Server Health Check")
print("=" * 60)

init_payload = json.dumps({
    "jsonrpc": "2.0", "id": 1, "method": "initialize",
    "params": {"protocolVersion": "2025-03-26", "capabilities": {},
               "clientInfo": {"name": "healthcheck", "version": "1.0"}}
})

for name, url in sorted(mcp_urls.items()):
    r = subprocess.run(
        ["curl", "-sk", "-X", "POST",
         "-H", "Content-Type: application/json",
         "-H", "Accept: application/json, text/event-stream",
         "-d", init_payload,
         "-o", "/dev/null", "-w", "%{http_code}", "-m", "5", url],
        capture_output=True, text=True)
    code = r.stdout.strip()
    status = "PASS" if code in ["200", "405"] else f"FAIL ({code})"
    print(f"  [{status:<8}] {name:<20} {url}")

print("")
print("IDE verification:")
print("  Cursor:     Settings > MCP > verify green indicators")
print("  VS Code:    Command Palette > 'MCP: List Servers'")
print("  Claude Code: /mcp to list connected servers")
print("  OpenCode:   Launch 'opencode' in project root > /mcp to list servers")

MCP Server Health Check
  [PASS    ] code-sandbox         https://mcp-code-sandbox-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp
  [PASS    ] codebase-search      https://mcp-codebase-search-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp
  [PASS    ] context7             https://mcp-context7-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp
  [PASS    ] searxng           https://mcp-searxng-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp
  [PASS    ] ocp-server           https://ocp-mcp-server-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp
  [PASS    ] repo-docs            https://mcp-repo-docs-mcp-servers.apps.openshift-cluster.sandbox1785.opentlc.com/mcp

IDE verification:
  Cursor:     Settings > MCP > verify green indicators
  VS Code:    Command Palette > 'MCP: List Servers'
  Claude Code: /mcp to list connected servers


## Next Steps

- `../3_basic_run/1_ide_model_config.ipynb` — Full IDE setup including **model endpoint** + **MaaS Gateway** configuration
- `../3_basic_run/2_run_public_coding_assistant.ipynb` — Run with all 5 tools (internet required)
- `../3_basic_run/3_run_closed_coding_assistant.ipynb` — Run with 3 local tools only (air-gapped)